# Erasus — CNN Coreset Ablation (CIFAR-10, Tesla T4)

This notebook runs a **coreset-based unlearning ablation** on a small CNN trained on CIFAR-10.

We train a model to memorize a subset of classes (the *forget set*), then use
[Erasus](https://github.com/OnePunchMonk/erasus) to surgically remove that
knowledge via **gradient ascent** while varying:

- **Coreset selector** — 7 methods (`influence`, `gradient_norm`, `el2n`, `tracin`, `herding`, `kcenter`, `random`)
- **Coreset fraction** — 9 values from 1% to 100% of the forget set

The result is a comprehensive picture of how selector quality and coreset size
affect the forget/retain accuracy trade-off.

**Runtime:** ~15–25 min on a free Colab T4 GPU.

In [ ]:
!pip install -q git+https://github.com/OnePunchMonk/erasus.git matplotlib torchvision
import torch
print(f"CUDA: {torch.cuda.is_available()}, Device: {torch.cuda.get_device_name(0)}")

In [ ]:
import os
import time
import json
import warnings
from copy import deepcopy

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------ #
# Model
# ------------------------------------------------------------------ #

class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, n_classes),
        )

    def forward(self, x, labels=None, **kwargs):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        logits = self.classifier(x)
        if labels is not None:
            return type("Out", (), {"logits": logits, "loss": F.cross_entropy(logits, labels)})()
        return logits

# ------------------------------------------------------------------ #
# Helpers
# ------------------------------------------------------------------ #

def compute_accuracy(model, loader, device):
    """Return accuracy (0–1) of *model* on *loader*."""
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            if hasattr(logits, "logits"):
                logits = logits.logits
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total if total > 0 else 0.0

# ------------------------------------------------------------------ #
# Data
# ------------------------------------------------------------------ #

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

cifar10 = datasets.CIFAR10(root="/content/data", train=True, download=True, transform=transform)

# Forget: 500 samples from classes 0, 1
# Retain: 1500 samples from classes 2–9
forget_classes = {0, 1}
forget_indices, retain_indices = [], []

for idx, (_, label) in enumerate(cifar10):
    if label in forget_classes and len(forget_indices) < 500:
        forget_indices.append(idx)
    elif label not in forget_classes and len(retain_indices) < 1500:
        retain_indices.append(idx)
    if len(forget_indices) >= 500 and len(retain_indices) >= 1500:
        break

print(f"Forget samples: {len(forget_indices)}, Retain samples: {len(retain_indices)}")

forget_set = Subset(cifar10, forget_indices)
retain_set = Subset(cifar10, retain_indices)

forget_loader = DataLoader(forget_set, batch_size=64, shuffle=True)
retain_loader = DataLoader(retain_set, batch_size=64, shuffle=True)

# Combine for training
train_set = Subset(cifar10, forget_indices + retain_indices)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)

# ------------------------------------------------------------------ #
# Train to memorize
# ------------------------------------------------------------------ #

model = SmallCNN(n_classes=10).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print("\nTraining for 20 epochs ...")
for epoch in range(1, 21):
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs, labels=labels)
        out.loss.backward()
        optimizer.step()
        running_loss += out.loss.item()
    if epoch % 5 == 0 or epoch == 1:
        print(f"  Epoch {epoch:>2d} | loss {running_loss / len(train_loader):.4f}")

base_forget_acc = compute_accuracy(model, forget_loader, device)
base_retain_acc = compute_accuracy(model, retain_loader, device)
print(f"\nBase model  —  Forget acc: {base_forget_acc:.4f} | Retain acc: {base_retain_acc:.4f}")

# Snapshot
trained_state = deepcopy(model.state_dict())

In [ ]:
from erasus import ErasusUnlearner

# ------------------------------------------------------------------ #
# Ablation grid
# ------------------------------------------------------------------ #

FRACTIONS = [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
SELECTORS = ["influence", "gradient_norm", "el2n", "tracin", "herding", "kcenter", "random"]

results = {}  # {selector: {fraction: {forget_acc, retain_acc}}}

total_runs = len(SELECTORS) * len(FRACTIONS)
run_idx = 0

for selector_name in SELECTORS:
    results[selector_name] = {}
    for frac in FRACTIONS:
        run_idx += 1
        tag = f"[{run_idx}/{total_runs}]"

        # Fresh copy of the trained model
        m = SmallCNN(n_classes=10).to(device)
        m.load_state_dict(deepcopy(trained_state))

        try:
            unlearner = ErasusUnlearner(
                model=m,
                strategy="gradient_ascent",
                selector=selector_name,
                selector_config={"fraction": frac},
            )
            t0 = time.time()
            unlearner.fit(
                forget_data=forget_loader,
                retain_data=retain_loader,
                epochs=5,
            )
            elapsed = time.time() - t0

            f_acc = compute_accuracy(m, forget_loader, device)
            r_acc = compute_accuracy(m, retain_loader, device)
        except Exception as exc:
            print(f"{tag}  {selector_name:>15s}  frac={frac:.2f}  —  ERROR: {exc}")
            f_acc = float("nan")
            r_acc = float("nan")
            elapsed = 0.0

        results[selector_name][frac] = {
            "forget_acc": f_acc,
            "retain_acc": r_acc,
            "time_s": round(elapsed, 2),
        }
        print(
            f"{tag}  {selector_name:>15s}  frac={frac:.2f}  |  "
            f"forget={f_acc:.4f}  retain={r_acc:.4f}  ({elapsed:.1f}s)"
        )

# ------------------------------------------------------------------ #
# Summary table
# ------------------------------------------------------------------ #

print("\n" + "=" * 90)
print(f"{'Selector':>15s}  {'Frac':>5s}  {'Forget':>8s}  {'Retain':>8s}  {'Time(s)':>8s}")
print("-" * 90)
for sel in SELECTORS:
    for frac in FRACTIONS:
        r = results[sel][frac]
        print(
            f"{sel:>15s}  {frac:>5.2f}  "
            f"{r['forget_acc']:>8.4f}  {r['retain_acc']:>8.4f}  {r['time_s']:>8.2f}"
        )
    print()
print("=" * 90)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ------------------------------------------------------------------ #
# Prepare arrays
# ------------------------------------------------------------------ #

fracs = np.array(FRACTIONS)

forget_curves = {}  # selector -> np array of forget accs
retain_curves = {}  # selector -> np array of retain accs
trade_curves  = {}  # selector -> np array of tradeoff score

for sel in SELECTORS:
    fa = np.array([results[sel][f]["forget_acc"] for f in FRACTIONS])
    ra = np.array([results[sel][f]["retain_acc"] for f in FRACTIONS])
    forget_curves[sel] = fa
    retain_curves[sel] = ra
    # Tradeoff: higher is better (maximize forgetting while keeping retain)
    trade_curves[sel] = (1.0 - fa) * ra

# ------------------------------------------------------------------ #
# 3-panel figure
# ------------------------------------------------------------------ #

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for sel in SELECTORS:
    axes[0].plot(fracs, forget_curves[sel], marker="o", label=sel)
    axes[1].plot(fracs, retain_curves[sel], marker="o", label=sel)
    axes[2].plot(fracs, trade_curves[sel], marker="o", label=sel)

axes[0].set_title("Forget Accuracy vs Coreset Fraction")
axes[0].set_xlabel("Coreset Fraction")
axes[0].set_ylabel("Forget Accuracy")
axes[0].axhline(base_forget_acc, ls="--", color="gray", alpha=0.5, label="base")
axes[0].legend(fontsize=7)
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Retain Accuracy vs Coreset Fraction")
axes[1].set_xlabel("Coreset Fraction")
axes[1].set_ylabel("Retain Accuracy")
axes[1].axhline(base_retain_acc, ls="--", color="gray", alpha=0.5, label="base")
axes[1].legend(fontsize=7)
axes[1].grid(True, alpha=0.3)

axes[2].set_title("Tradeoff Score  (1 - forget) * retain")
axes[2].set_xlabel("Coreset Fraction")
axes[2].set_ylabel("Tradeoff Score")
axes[2].legend(fontsize=7)
axes[2].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig("/content/coreset_ablation_3panel.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved /content/coreset_ablation_3panel.png")

# ------------------------------------------------------------------ #
# Single retain-accuracy plot
# ------------------------------------------------------------------ #

fig2, ax2 = plt.subplots(figsize=(8, 5))
for sel in SELECTORS:
    ax2.plot(fracs, retain_curves[sel], marker="s", linewidth=2, label=sel)
ax2.axhline(base_retain_acc, ls="--", color="gray", alpha=0.6, label="base retain")
ax2.set_title("Retain Accuracy After Unlearning")
ax2.set_xlabel("Coreset Fraction")
ax2.set_ylabel("Retain Accuracy")
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

fig2.tight_layout()
fig2.savefig("/content/retain_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved /content/retain_accuracy.png")

In [ ]:
# ------------------------------------------------------------------ #
# Save results JSON and download
# ------------------------------------------------------------------ #

output = {
    "experiment": "cnn_cifar10_coreset_ablation",
    "model": "SmallCNN",
    "dataset": "CIFAR-10",
    "forget_classes": [0, 1],
    "forget_samples": len(forget_indices),
    "retain_samples": len(retain_indices),
    "train_epochs": 20,
    "unlearn_epochs": 5,
    "strategy": "gradient_ascent",
    "base_forget_acc": base_forget_acc,
    "base_retain_acc": base_retain_acc,
    "fractions": FRACTIONS,
    "selectors": SELECTORS,
    "results": {},
}

for sel in SELECTORS:
    output["results"][sel] = {}
    for frac in FRACTIONS:
        r = results[sel][frac]
        output["results"][sel][str(frac)] = {
            "forget_acc": round(r["forget_acc"], 6) if not np.isnan(r["forget_acc"]) else None,
            "retain_acc": round(r["retain_acc"], 6) if not np.isnan(r["retain_acc"]) else None,
            "time_s": r["time_s"],
        }

json_path = "/content/coreset_ablation_results.json"
with open(json_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Results saved to {json_path}")

# Download files (Colab-specific)
try:
    from google.colab import files
    files.download(json_path)
    files.download("/content/coreset_ablation_3panel.png")
    files.download("/content/retain_accuracy.png")
except ImportError:
    print("Not running on Colab — files saved locally under /content/")